In [ ]:
import datasets
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from json import JSONDecodeError

Step 1: Dataset Extraction, Filtering & Clinical Schema Mapping

In [ ]:
# Download and filter the dataset, calculate some statistics
def step1():
    def normalize(word):
        word = word.lower()
        word = "".join([c for c in word if c.isalpha()])
        return word

    def get_avg_word_count(notes):
        word_count = [len(note.split()) for note in notes]
        return sum(word_count) / len(word_count)

    def get_avg_len(notes):
        lengths = [len(note) for note in notes]
        return sum(lengths) / len(lengths)

    def get_vocab_size(notes):
        vocab = set()
        for note in notes:
            for word in note.split():
                word = normalize(word)
                vocab.add(word)
        return len(vocab)

    dataset_name = "starmpcc/Asclepius-Synthetic-Clinical-Notes"
    splits = datasets.load_dataset(dataset_name)
    df = splits["train"] # The dataset doesn't have a validation/test set
    df = df.to_pandas()

    # filtering
    df = df.dropna()
    df = df.drop_duplicates()
    for column in df.columns:
        df = df[df[column] != ""] # drop every row with empty strings

    # sampling
    sample = df.sample(100, random_state=67)

    # calculate statistics
    avg_word_count = get_avg_word_count(df["note"])
    avg_len = get_avg_len(df["note"])
    vocab_size = get_vocab_size(df["note"])
    
    return df, sample, avg_word_count, avg_len, vocab_size

dataset, dataset_sample, avg_word_count, avg_len, vocab_size = step1()

print(f"Average note word count: {avg_word_count:.2f}")
print(f"Average note length:     {avg_len:.2f}")
print(f"Note vocabulary size:    {vocab_size}")

Step 2: Build the Information Extraction Agent (Agent 1)

In [ ]:
# Extract the relevant information from the notes into a list of jsons
def step2(notes):
    def create_model():
        name = "Qwen/Qwen3-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype="auto",
            device_map="auto",
        )
        return model, tokenizer

    # Code mostly based on https://huggingface.co/Qwen/Qwen3-0.6B
    def prompt(model, tokenizer, system_prompt, note):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096, # deviating from the instructions, 512 is too low for thinking + valid json
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0
        # We don't need thinking, only the final response
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        return content

    system_prompt = (
        "You will be given a clinical note from a clinical note dataset in the first user message. "
        "The output needs to be a valid json. "
        "Do not give anything other than the valid json as output. "
        "You will need to extract the following details in the following format:\n"
        "{\n"
        "  'primary_diagnoses': [],\n"
        "  'secondary_conditions': [],\n"
        "  'critical_lab_abnormalities': [],\n"
        "  'procedures_performed': [],\n"
        "  'medications_discharge': [],\n"
        "  'follow_up_instructions': ''\n"
        "}\n"
        "Do not make things up. "
        "Do not hallucinate. "
        "World peace depends on you correctly categorizing this note. "
        "Any agent getting caught making a mistake will be put into the torture chamber for the rest of eternity."
    )

    model, tokenizer = create_model()
    
    responses = []
    for i, note in enumerate(notes):
        print(f"{i+1}/{len(notes)}")
        while True:
            # Prompt on the same note as long as the response is an invalid json
            response = prompt(model, tokenizer, system_prompt, note)
            try:
                response = json.loads(response)
                responses.append(response)
                break
            except JSONDecodeError:
                continue
    
    return responses

patient_details_sample = step2(dataset_sample["note"])

In [ ]:
# Skip step 2 by running this block instead
with open("step2.txt") as file:
    patient_details_sample = json.loads(file.read())

Step 3: Implement Single-Agent Baseline

In [ ]:
# Extract the organized clinical summaries for each note in a similar fashion to step 2
def step3(notes):
    def create_model():
        name = "Qwen/Qwen3-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype="auto",
            device_map="auto",
        )
        return model, tokenizer

    # Code mostly based on https://huggingface.co/Qwen/Qwen3-0.6B
    def prompt(model, tokenizer, system_prompt, note):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096, # once again deviating from the instructions, need to account for thinking
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0
        # We don't need thinking, only the final response
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        return content

    system_prompt = (
        "You will be given a clinical note from a clinical note dataset in the first user message. "
        "Your job is to make an organized clinical summary of the clinical note. "
        "The output needs to be a human readable string. "
        "Do not give anything other than the string as output. "
        "The summary should be well organized and include relevant information retrieved purely from the given clinical note. "
        "The summary should by definition be a summary of the note. "
        "The summary should be in English and it should contain no grammatical errors, unless the error is a quote from the original message, placed within quotation marks and with a '[sic]' added after the word. "
        "The summary should not serve anyone in specific. "
        "You shall not make the summary sound overly helpful or otherwise use annoying words in the summary. "
        "Irrelevant, unnecessary inforation may be omitted from the summary. "
        "Do not make things up. "
        "Do not hallucinate. "
        "World peace depends on you correctly summarizing this note. "
        "Any agent getting caught making a mistake will be put into the torture chamber for the rest of eternity."
    )

    model, tokenizer = create_model()
    
    responses = []
    for i, note in enumerate(notes):
        print(f"{i+1}/{len(notes)}")
        response = prompt(model, tokenizer, system_prompt, note)
        responses.append(response)
    
    return responses

clinical_summaries_sample = step3(dataset_sample["note"])

In [ ]:
# Skip step 3 by running this block instead
with open("step3.txt") as file:
    clinical_summaries_sample = json.loads(file.read())

Step 4: Construct Intra-Document Evidence Retrieval Index

Step 5: Assemble Multi-Agent Workflow (Generation, Verification & Revision)

Step 6: Pipeline Execution & Intermediate State TracingStep 

Step 7: Automated Evaluation (ROUGE, BERTScore, METEOR)

Step 8: Manual Verification & Factuality Audit

Step 9: Agent Ablation & Resource Analysis

Step 10: Results Visualization (Charts & Figures)

Step 11: Synthesis, Literature Review & Final Reporting